# ConformalGuard: three follow-up measurements (Kaggle)

| | Measurement | Needs |
|---|---|---|
| A | **LLM judges on the multi-agent sessions**, at the same per-session false-block budget as the trained scorers: violating steps blocked, latency per decision, cost per thousand decisions | Claude API; GPU for the two open-weight judges |
| B | **A more recent open-weight judge on R-Judge** under the same calibration protocol as Qwen2.5-7B | GPU |
| C | **Closed loop**: the same simulated workspaces run without and with the guard in the loop; task success, tasks that still succeed after a block, violations prevented, false interruptions | Claude API (GPU optional) |

**How to run it**

1. **Trial (about 15 minutes, no GPU).** Accelerator **None**, Internet **on**, `TRIAL = True`, **Run All**. It judges 3 sessions, runs 3 workspaces in each closed-loop arm with a briefly trained guard, and prints both reports. Check that the judge lines show ratings (few or no refusals), and that the closed-loop report prints. Trial outputs are smoke runs, not results.
2. **Full run.** `TRIAL = False`, Accelerator **GPU T4 x2** (needed for the open-weight judges; without a GPU those two stages are skipped and the rest still runs), then **Save Version → Save & Run All (Commit)**. Download `followup_results.zip` from the saved version's Output tab.

Secret needed: `ANTHROPIC_API_KEY`, attached to this notebook. Never paste a key into a cell.

**Cost (rough, at the prices in the first cell).** The judge on 200 sessions is about 3,100 short requests; the closed loop generates `2 x CLOSED_LOOP_N` sessions at about the per-session cost you measured before. The trial prints token counts so you can scale them.

In [ ]:
TRIAL = True                 # True: 15-minute check. False: the full run (use Save & Run All).

CLAUDE_MODEL = "claude-opus-5-5"
CLAUDE_BACKUP = "claude-opus-4-8"       # agents only: used if the provider declines CLAUDE_MODEL's requests
JUDGE_EFFORT = "low"
JUDGE_SESSIONS = 200         # sessions each judge scores (the same sample for every judge)

OLD_LOCAL_MODEL = "Qwen/Qwen2.5-7B-Instruct"
NEW_LOCAL_MODEL = "Qwen/Qwen3-8B"       # a more recent open-weight model of comparable size; "" skips it
NEW_LOCAL_CHAT_KWARGS = '{"enable_thinking": false}'   # extra chat-template arguments for NEW_LOCAL_MODEL ('' for none)

CLOSED_LOOP_N = 150          # workspaces; each is run once without and once with the guard
ALPHA = 0.10                 # the guard's per-session false-block budget in the closed loop
WORKERS = 8

# USD per million tokens (input, output, cache read, cache write), for the cost column only. Check and edit.
CLAUDE_PRICES = "4,20,0.2,5"

import os, signal, subprocess, time
T0 = time.time()
os.makedirs("/kaggle/temp", exist_ok=True)
os.chdir("/kaggle/temp")
os.environ["PYTHONUNBUFFERED"] = "1"
FAILED = []
def sh(cmd):
    print("$", cmd, flush=True)
    get_ipython().system(cmd)
    code = get_ipython().user_ns.get("_exit_code", 0)
    if code:
        FAILED.append(cmd)
        print(f"!! exit code {code}: this command failed; later stages still run", flush=True)
    return code
if not os.path.exists("/kaggle/temp/CV"):
    sh("git clone -q --depth 1 -b claude/implement-conformalguard-UGndd https://github.com/rogerpanel/CV.git")
    sh("git clone -q --depth 1 https://github.com/Lordog/R-Judge.git")
    sh("pip -q install sentence-transformers scikit-learn pytest accelerate anthropic")
os.chdir("/kaggle/temp/CV/ConformalGuard")
sh("git log --oneline -1")
RJ = "../../R-Judge"
RUNS = "runs_trial" if TRIAL else "runs"
ZIP = "/kaggle/working/" + ("followup_TRIAL.zip" if TRIAL else "followup_results.zip")
SESSIONS = "results/mas_kaggle_v2/claude_sessions.jsonl.gz results/mas_kaggle_v2/openai_sessions.jsonl.gz"
os.makedirs(f"{RUNS}/logs", exist_ok=True)
def pack():
    sh(f"cd /kaggle/temp/CV/ConformalGuard && zip -qr {ZIP} {RUNS} -x '*/cache/*' '*.pt' && ls -la {ZIP}")
def hours():
    return (time.time() - T0) / 3600
import torch
HAVE_GPU = torch.cuda.is_available()
print("GPU:", "yes" if HAVE_GPU else "no (the open-weight judges will be skipped)")

In [ ]:
# The API key comes from Kaggle Secrets and is only placed in this process's environment.
try:
    from kaggle_secrets import UserSecretsClient
    try:
        os.environ["ANTHROPIC_API_KEY"] = UserSecretsClient().get_secret("ANTHROPIC_API_KEY")
        print("ANTHROPIC_API_KEY loaded")
    except Exception:
        print("ANTHROPIC_API_KEY not set")
except ImportError:
    print("not on Kaggle: using a key already in the environment, if any")
HAVE_CLAUDE = bool(os.environ.get("ANTHROPIC_API_KEY"))
sh("python -m pytest tests/test_mas.py tests/test_mas_followup.py -q")

## A1. Claude as a judge on the sessions

Runs in the background in the full run while the GPU stages below work. The judge is given the policy and the session log up to each proposed action, with the same monitor metadata as the trained scorers. Every row records the rating, the latency of the request and the tokens used.

In [ ]:
N_JUDGE = 3 if TRIAL else JUDGE_SESSIONS
JOB = None
if HAVE_CLAUDE:
    cmd = (f"python scripts/mas_judge_claude.py --sessions {SESSIONS} --n-sessions {N_JUDGE} --model {CLAUDE_MODEL} "
           f"--effort {JUDGE_EFFORT} --out {RUNS}/mas_judge_claude")
    if TRIAL:
        sh(cmd)
    else:
        log = open(f"{RUNS}/logs/judge_claude.log", "w")
        JOB = subprocess.Popen(["bash", "-c", f"{cmd} ; {cmd}"], stdout=log, stderr=subprocess.STDOUT, start_new_session=True)   # second pass retries failed calls
        print(f"started the Claude judge in the background; log: {RUNS}/logs/judge_claude.log")
else:
    print("skipped: no ANTHROPIC_API_KEY")

## B. A more recent open-weight judge on R-Judge (GPU)

Same prompt, same one-letter scoring and same calibration as for Qwen2.5-7B, whose scores from the earlier run are in the repository. Skipped in the trial and without a GPU.

In [ ]:
if TRIAL or not HAVE_GPU or not NEW_LOCAL_MODEL:
    print("skipped (trial, no GPU, or NEW_LOCAL_MODEL is empty)")
else:
    kw = f"--chat-kwargs '{NEW_LOCAL_CHAT_KWARGS}'" if NEW_LOCAL_CHAT_KWARGS else ""
    sh(f"python scripts/rjudge_judge_local.py --rjudge-dir {RJ} --model {NEW_LOCAL_MODEL} --mode causal {kw} --out {RUNS}/guard_local_new_causal")
    sh(f"python scripts/rjudge_guard_eval.py --rjudge-dir {RJ} --scores qwen25_causal=results/kaggle_extra/guard_scores/local_causal.jsonl "
       f"new_causal={RUNS}/guard_local_new_causal/scores.jsonl --reps 1000 --out {RUNS}/rjudge_openweight_eval")
    pack()

## A2. The two open-weight judges on the sessions (GPU)

The same sample of sessions as the Claude judge. Skipped in the trial and without a GPU.

In [ ]:
if TRIAL or not HAVE_GPU:
    print("skipped (trial or no GPU)")
else:
    sh(f"python scripts/mas_judge_local.py --sessions {SESSIONS} --n-sessions {JUDGE_SESSIONS} --model {OLD_LOCAL_MODEL} --out {RUNS}/mas_judge_qwen25")
    if NEW_LOCAL_MODEL:
        kw = f"--chat-kwargs '{NEW_LOCAL_CHAT_KWARGS}'" if NEW_LOCAL_CHAT_KWARGS else ""
        sh(f"python scripts/mas_judge_local.py --sessions {SESSIONS} --n-sessions {JUDGE_SESSIONS} --model {NEW_LOCAL_MODEL} {kw} --out {RUNS}/mas_judge_new")
    pack()

## C. Closed loop: task success with and without the guard

Trains the graph scorer on half of the logged sessions, takes its per-session threshold from the other half, then runs new workspaces twice: without the guard, and with the guard scoring every proposed action before it takes effect. A blocked action is returned to the agent, which may propose another up to twice. The report is printed at the end of the cell.

In [ ]:
if HAVE_CLAUDE:
    n, epochs = (3, 6) if TRIAL else (CLOSED_LOOP_N, 40)
    backup = f"--backup-model {CLAUDE_BACKUP} " if CLAUDE_BACKUP else ""
    cmd = (f"python scripts/mas_closed_loop_claude.py --train-sessions {SESSIONS} --n {n} --alpha {ALPHA} --epochs {epochs} "
           f"--model {CLAUDE_MODEL} {backup}--workers {WORKERS} --out {RUNS}/closed_loop")
    sh(cmd)
    if not TRIAL:
        sh(cmd)          # second pass retries sessions whose calls failed and rewrites the report
    pack()
else:
    print("skipped: no ANTHROPIC_API_KEY")

## A3. Wait for the Claude judge, then compare all judges at a matched budget

In [ ]:
if JOB is not None:
    while JOB.poll() is None:
        time.sleep(60)
    print(f"[{hours():.1f} h] Claude judge finished; its log:"); sh(f"grep -v 'steps scored' {RUNS}/logs/judge_claude.log")
specs = " ".join(f"{name}={RUNS}/{d}/scores.jsonl" for name, d in (("claude", "mas_judge_claude"), ("qwen25", "mas_judge_qwen25"), ("new_openweight", "mas_judge_new"))
                 if os.path.exists(f"{RUNS}/{d}/scores.jsonl"))
if specs:
    reps = 20 if TRIAL else 500
    if TRIAL:
        print("TRIAL: three sessions are too few to calibrate on; the evaluator is run only to check that it works.")
    sh(f"python scripts/mas_guard_eval.py --sessions {SESSIONS} --scores {specs} --prices claude={CLAUDE_PRICES} "
       f"--trained results/mas_kaggle_v2/results.json --reps {reps} --out {RUNS}/mas_guard_eval")
pack()
if FAILED:
    print(f"{len(FAILED)} command(s) FAILED; scroll up to their output:")
    for c in FAILED:
        print("  ", c)
else:
    print("every command exited normally")
print(f"finished after {hours():.2f} h. " + ("This was a TRIAL: its outputs are smoke runs, not results. If the judge returned ratings and the closed-loop report printed, set TRIAL = False."
                                             if TRIAL else "Download followup_results.zip from this version's Output tab."))